# M.Tech AIML – Machine Learning Assignment 2
## Dry Bean Multi-class Classification

This notebook implements five classifiers on the UCI Dry Bean dataset and evaluates them using the six metrics required in the assignment.

**Dataset file required:** `SampleDataset.csv` in the same folder as this notebook.

**Target:** `Class` (7 bean categories). **Inputs:** 16 numerical features.

Run the cells from top to bottom. The notebook checks the dataset schema before training.

## 1. Import libraries

In [3]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)

RANDOM_STATE = 42
DATA_PATH = "SampleDataset.csv"
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Libraries imported successfully.")

ModuleNotFoundError: No module named 'joblib'

## 2. Load and validate the dataset

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Put SampleDataset.csv in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

expected_features = [
    "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRation", "Eccentricity", "ConvexArea", "EquivDiameter",
    "Extent", "Solidity", "roundness", "Compactness",
    "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"
]
target_column = "Class"

missing_columns = [c for c in expected_features + [target_column] if c not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

if df.shape != (13611, 17):
    warnings.warn(
        f"Expected the full UCI Dry Bean dataset shape (13611, 17), but found {df.shape}. "
        "Continuing only if the required columns and minimum assignment size are satisfied."
    )

if df.shape[0] < 500 or len(expected_features) < 12:
    raise ValueError("Dataset does not meet the assignment minimum of 500 rows and 12 features.")

if df[expected_features].isnull().any().any() or df[target_column].isnull().any():
    raise ValueError("Missing values found. Resolve them before training.")

# Ensure all predictor columns are numeric
for col in expected_features:
    df[col] = pd.to_numeric(df[col], errors="raise")

X = df[expected_features].copy()
y = df[target_column].astype(str).copy()

print("Dataset shape:", df.shape)
print("Number of input features:", X.shape[1])
print("Number of classes:", y.nunique())
print("Class names:", sorted(y.unique()))
print("Missing values:", int(df.isnull().sum().sum()))
display(df.head())

## 3. Split into training and testing data
We use a stratified 80:20 split so each class is represented proportionally in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

## 4. Define the five required classification models
Scaling is included in pipelines for Logistic Regression, KNN and SVM. Decision Tree and Gaussian Naive Bayes use the numeric features directly.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Gaussian Naive Bayes": GaussianNB(),
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(probability=True, random_state=RANDOM_STATE))
    ])
}

print("Models defined:", list(models.keys()))

## 5. Train models and calculate all six evaluation metrics
AUC is calculated as weighted one-vs-rest (OvR), appropriate for this 7-class classification problem.

In [ ]:
results = []
trained_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    print(f"Training {name} ...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    # Align probability columns with the model's learned class order
    auc = roc_auc_score(
        y_test,
        y_prob,
        labels=model.classes_,
        multi_class="ovr",
        average="weighted"
    )

    metrics = {
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": auc,
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

    results.append(metrics)
    trained_models[name] = model
    predictions[name] = y_pred
    probabilities[name] = y_prob

    print(classification_report(y_test, y_pred, zero_division=0))

comparison_df = pd.DataFrame(results)
display(comparison_df.style.format({
    "Accuracy": "{:.4f}", "AUC": "{:.4f}", "Precision": "{:.4f}",
    "Recall": "{:.4f}", "F1 Score": "{:.4f}", "MCC": "{:.4f}"
}))

## 6. Model observations and best model
The best model below is selected by test-set accuracy. Review all metrics before writing your final observation; do not assume the same model wins every metric.

In [ ]:
best_row = comparison_df.loc[comparison_df["Accuracy"].idxmax()]
print("Best model by test accuracy:", best_row["ML Model Name"])
print("Accuracy:", round(best_row["Accuracy"], 4))
print("\nComparison table:")
display(comparison_df.sort_values("Accuracy", ascending=False).reset_index(drop=True))

## 7. Confusion matrices
Each confusion matrix compares actual and predicted bean classes for the test set.

In [ ]:
class_labels = sorted(y.unique())

for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_labels, yticklabels=class_labels
    )
    plt.title(f"Confusion Matrix – {name}")
    plt.xlabel("Predicted class")
    plt.ylabel("Actual class")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 8. Save trained models and evaluation results
The saved models include preprocessing pipelines where required, so the Streamlit app can use the same scaling at prediction time.

In [ ]:
for name, model in trained_models.items():
    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )
    joblib.dump(model, os.path.join(MODEL_DIR, f"{safe_name}.joblib"))

# Save feature order and class labels for the application
metadata = {
    "feature_columns": expected_features,
    "target_column": target_column,
    "class_labels": class_labels,
    "random_state": RANDOM_STATE
}
joblib.dump(metadata, os.path.join(MODEL_DIR, "metadata.joblib"))

comparison_df.to_csv("model_comparison.csv", index=False)

print("Saved five trained models and metadata in the 'model' folder.")
print("Saved model_comparison.csv")

## 9. Export test data for the Streamlit application
The CSV includes all 16 input features and the actual `Class` target. The app can calculate evaluation metrics without retraining.

In [ ]:
test_data = X_test.copy()
test_data[target_column] = y_test.values
test_data.to_csv("test_data.csv", index=False)

print("test_data.csv created.")
print("Test data shape:", test_data.shape)
display(test_data.head())

## 10. Final verification checklist

In [ ]:
required_outputs = [
    "test_data.csv",
    "model_comparison.csv",
    os.path.join(MODEL_DIR, "logistic_regression.joblib"),
    os.path.join(MODEL_DIR, "decision_tree.joblib"),
    os.path.join(MODEL_DIR, "k_nearest_neighbors.joblib"),
    os.path.join(MODEL_DIR, "gaussian_naive_bayes.joblib"),
    os.path.join(MODEL_DIR, "support_vector_machine.joblib"),
    os.path.join(MODEL_DIR, "metadata.joblib")
]

for path in required_outputs:
    print(("OK      " if os.path.exists(path) else "MISSING "), path)

assert len(trained_models) == 5
assert X.shape[1] == 16
assert y.nunique() == 7
assert test_data.shape[1] == 17

print("\nNotebook workflow completed successfully.")